# LTX Video Test: Text-to-Video + Image-to-Video

Prueba para verificar [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video) (modelo 2B real, ~10 GB en bf16).

Usa `device_map="balanced"` para que accelerate cargue cada componente directo a GPU y libere la copia en RAM, evitando OOM en 12 GB RAM.

In [ ]:
# ---- 1. INSTALAR DEPENDENCIAS ----
import os, torch
!pip install -qU diffusers transformers accelerate sentencepiece
!pip install -q imageio[ffmpeg] pillow psutil

In [ ]:
# ---- 2. IMPORTS + DIAGNOSTICO ----
import torch, gc, psutil
from diffusers import LTXPipeline
from diffusers.utils import load_image, export_to_video
from IPython.display import Video

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16

ram = psutil.virtual_memory()
print(f'RAM total: {ram.total/1024**3:.1f} GB, disponible: {ram.available/1024**3:.1f} GB')
if DEVICE == 'cuda':
    gpu = torch.cuda.get_device_name()
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu} ({vram:.1f} GB VRAM)')

## Text-to-Video

Carga con `device_map="balanced"` para evitar OOM en RAM de sistema.

In [ ]:
# ---- 3. TEXT-TO-VIDEO ----

gc.collect(); torch.cuda.empty_cache()
ram_avail = psutil.virtual_memory().available / 1024**3
print(f'RAM disponible: {ram_avail:.1f} GB')

if ram_avail < 3:
    print('ERROR: memoria insuficiente')
else:
    print('Cargando LTXPipeline con device_map="balanced"...')
    pipe = LTXPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=DTYPE,
        device_map='balanced',
    )
    pipe.vae.enable_tiling()
    pipe.enable_attention_slicing()
    print('OK')

    prompt = 'A cinematic drone shot of a misty mountain range at sunrise.'

    with torch.inference_mode():
        output = pipe(
            prompt=prompt,
            width=512, height=288,
            num_frames=9,
            num_inference_steps=20,
            guidance_scale=5.0,
        )

    export_to_video(output.frames[0], '/content/ltx_t2v.mp4', fps=24)
    print('Video: /content/ltx_t2v.mp4')
    Video('/content/ltx_t2v.mp4', embed=True, width=512)

    del pipe; gc.collect(); torch.cuda.empty_cache()

## Image-to-Video

Misma configuración, pero condicionado con una imagen inicial.

In [ ]:
# ---- 4. IMAGE-TO-VIDEO ----

gc.collect(); torch.cuda.empty_cache()
if psutil.virtual_memory().available / 1024**3 < 3:
    print('ERROR: memoria insuficiente')
else:
    print('Cargando LTXConditionPipeline...')
    from diffusers import LTXConditionPipeline
    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition

    pipe = LTXConditionPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=DTYPE,
        device_map='balanced',
    )
    pipe.vae.enable_tiling()
    pipe.enable_attention_slicing()

    image = load_image('https://huggingface.co/datasets/huggingface/documentation-images/'
                        'resolve/main/diffusers/guitar-man.png')
    condition = LTXVideoCondition(image=image, frame_index=0)

    with torch.inference_mode():
        output = pipe(
            prompt='A man playing guitar, slow motion.',
            conditions=[condition],
            width=512, height=288,
            num_frames=9,
            num_inference_steps=20,
            guidance_scale=5.0,
        )

    export_to_video(output.frames[0], '/content/ltx_i2v.mp4', fps=24)
    print('Video: /content/ltx_i2v.mp4')
    Video('/content/ltx_i2v.mp4', embed=True, width=512)

    del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 5. LIMPIEZA ----
gc.collect(); torch.cuda.empty_cache()
print('Done')